<a href="https://colab.research.google.com/github/Rus18mc/nash/blob/main/%D0%9A%D0%BE%D0%BF%D0%B8%D1%8F_%D0%B1%D0%BB%D0%BE%D0%BA%D0%BD%D0%BE%D1%82%D0%B0_%22gpr_rus_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



> Данные (D1 — daily timeframe) по акциям российских публичных компаний.

Каждый файл содержит стандартные биржевые поля:

datetime — дата торгов

high — максимальная цена за день

low — минимальная цена за день

close — цена закрытия

Период:

1999-06-01 - 2024-08-27





> Индикатор GPRD

Файл: stock_data.csv

Содержит:

dt — дата

GPRD — значение индекса

Период:

1990-01-03 - 2024-02-16




> Импорт библиотек



In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from google.colab import files
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('/content/stock_data.csv')
df

,dt,vix,sp500,sp500_volume,djia,djia_volume,hsi,ads,us3m,joblessness,epu,GPRD,prev_day
0,1990-01-03,18.19,358.760010,1.923300e+08,2809.73,23.62,2858.699951,-0.229917,7.89,3,100.359178,75.408051,359.690002
1,1990-01-04,19.22,355.670013,1.770000e+08,2796.08,24.37,2868.000000,-0.246065,7.84,3,100.359178,56.085804,358.760010
2,1990-01-05,20.11,352.200012,1.585300e+08,2773.25,20.29,2839.899902,-0.260393,7.79,3,100.359178,63.847675,355.670013
3,1990-01-08,20.26,353.790009,1.401100e+08,2794.37,16.61,2816.000000,-0.291750,7.79,3,100.359178,102.841156,352.200012
4,1990-01-09,22.20,349.619995,1.552100e+08,2766.00,15.80,2822.000000,-0.297326,7.80,3,100.359178,138.435669,353.790009
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8592,2024-02-12,13.93,5021.840000,3.805740e+09,38797.90,260.66,15746.580078,-0.092373,5.43,1,89.218778,156.723633,5026.610000
8593,2024-02-13,15.85,4953.170000,4.302190e+09,38272.49,305.80,15746.580078,-0.071135,5.45,1,89.218778,187.858292,5021.840000
8594,2024-02-14,14.38,5000.620000,3.845600e+09,38423.68,265.90,15879.379883,-0.052492,5.43,1,89.218778,135.809372,4953.170000
8595,2024-02-15,14.01,5029.730000,4.137970e+09,38773.12,300.35,15944.629883,-0.036436,5.43,1,89.218778,214.808151,5000.620000



>Чтение данных по двум акциям (GAZP и TATN). Приведение столбца datetime к типу datetime.




In [ ]:
df_gazp_original = pd.read_csv('/content/GAZP_D1.csv')
df_tatn_original = pd.read_csv('/content/TATN_D1.csv')
df_gazp_original['datetime'] = pd.to_datetime(df_gazp_original['datetime'])
df_tatn_original['datetime'] = pd.to_datetime(df_tatn_original['datetime'])





> Выбор нужных колонок: 'datetime', 'high', 'low', 'close'. Объединение GAZP и TATN по датам (inner join).



In [ ]:
df_gazp_data = df_gazp_original[['datetime', 'high', 'low', 'close']].copy()
df_gazp_data.rename(columns={'high': 'high_GAZP', 'low': 'low_GAZP', 'close': 'close_GAZP'}, inplace=True)

df_tatn_data = df_tatn_original[['datetime', 'high', 'low', 'close']].copy()
df_tatn_data.rename(columns={'high': 'high_TATN', 'low': 'low_TATN', 'close': 'close_TATN'}, inplace=True)


df_combined_high_low = pd.merge(df_gazp_data, df_tatn_data, on='datetime', how='inner')





> Добавление акций лучших российских металлургических и нефтегазовых компаний.


> Цикл с чтением, подготовкой и объединением каждой акции (inner join)



> Сбор одного общего датафрейма со свечами всех акций на пересечении дат (общие дни для всех акций).






In [ ]:
import pandas as pd
from functools import reduce

tickers = ['GAZP', 'TATN', 'CHMF', 'GMKN', 'LKOH', 'NLMK', 'ROSN', 'SNGS']
data_dir = "/content"
cutoff_date = pd.to_datetime("2024-02-16")

def load_ticker_d1(ticker: str, data_dir: str) -> pd.DataFrame:
    """Читает /content/<TICKER>_D1.csv и возвращает datetime + high/low/close с суффиксом тикера."""
    path = f"{data_dir}/{ticker}_D1.csv"
    df = pd.read_csv(path)
    df["datetime"] = pd.to_datetime(df["datetime"])
    df = df[["datetime", "high", "low", "close"]].copy()
    return df.rename(columns={
        "high": f"high_{ticker}",
        "low": f"low_{ticker}",
        "close": f"close_{ticker}",
    })




>Фильтр по дате



In [ ]:
dfs = [load_ticker_d1(t, data_dir) for t in tickers]
df_combined_high_low = reduce(
    lambda left, right: pd.merge(left, right, on="datetime", how="inner"),
    dfs
)




> Выводим начало и конец датафрейма чтобы визуально проверить, что объединение и фильтр сработали корректно.



In [ ]:
display(df_combined_high_low.head())
display(df_combined_high_low.tail())

,datetime,high_GAZP,low_GAZP,close_GAZP,high_TATN,low_TATN,close_TATN,high_CHMF,low_CHMF,close_CHMF,...,close_LKOH,high_NLMK,low_NLMK,close_NLMK,high_ROSN,low_ROSN,close_ROSN,high_SNGS,low_SNGS,close_SNGS
0,2008-05-23,2.175,2.113,2.140,188.0,184.1,187.5,605.0,584.8,594.6,...,5.055,418.0,365.0,367.0,270.97,261.71,269.82,29.450,28.590,29.115
1,2008-05-26,2.169,2.129,2.143,189.5,181.1,184.0,594.6,587.6,593.6,...,5.000,414.0,365.0,398.0,276.30,268.20,275.15,29.285,28.465,28.720
2,2008-05-27,2.151,2.095,2.120,186.5,181.2,182.1,597.0,583.8,589.2,...,5.040,400.0,400.0,400.0,281.65,270.50,272.75,29.515,27.545,27.675
3,2008-05-30,2.149,2.100,2.115,189.9,184.8,189.4,632.4,610.4,617.6,...,5.102,399.0,300.0,370.0,287.03,278.63,286.78,29.268,27.811,29.063
4,2008-06-03,2.185,2.113,2.177,188.5,185.0,186.9,612.6,590.8,596.0,...,5.250,411.0,378.0,378.0,287.65,282.60,283.75,28.952,28.420,28.767


,datetime,high_GAZP,low_GAZP,close_GAZP,high_TATN,low_TATN,close_TATN,high_CHMF,low_CHMF,close_CHMF,...,close_LKOH,high_NLMK,low_NLMK,close_NLMK,high_ROSN,low_ROSN,close_ROSN,high_SNGS,low_SNGS,close_SNGS
2510,2024-08-21,0.6001,0.5905,0.5993,605.0,585.9,603.5,1343.8,1310.4,1321.0,...,8.620,4520.0,4450.0,4450.0,485.35,478.50,480.95,25.815,25.305,25.745
2511,2024-08-22,0.6013,0.5735,0.5778,606.3,595.4,601.7,1328.0,1285.6,1292.8,...,8.375,4450.0,4120.0,4160.0,482.50,474.10,475.80,25.970,24.525,24.715
2512,2024-08-23,0.5800,0.5397,0.5520,605.9,586.5,596.6,1298.0,1234.6,1253.2,...,8.320,4210.0,3940.0,3980.0,479.25,470.60,478.75,25.120,23.505,24.275
2513,2024-08-26,0.5750,0.5565,0.5704,616.6,600.5,607.8,1319.0,1276.0,1318.6,...,8.475,4200.0,4100.0,4150.0,490.15,481.10,489.55,25.055,24.260,24.900
2514,2024-08-27,0.5800,0.5510,0.5545,616.0,606.3,611.6,1349.2,1265.4,1275.2,...,8.270,4200.0,4120.0,4120.0,491.45,475.75,477.50,25.200,24.180,24.250




>Добавляем GPRD из stock_data.csv и заполняем пропуски предыдущим значением (ffill)



In [ ]:
df_gprd = pd.read_csv(f"{data_dir}/stock_data.csv")
df_gprd["dt"] = pd.to_datetime(df_gprd["dt"])
df_gprd = df_gprd[["dt", "GPRD"]].rename(columns={"dt": "datetime"}).sort_values("datetime")

df_combined_high_low = (
    df_combined_high_low
    .merge(df_gprd, on="datetime", how="left")
    .sort_values("datetime")
)

df_combined_high_low["GPRD"] = df_combined_high_low["GPRD"].ffill()





> Фильтр по дате



In [ ]:
df_combined_high_low = df_combined_high_low[df_combined_high_low["datetime"] <= cutoff_date].copy()




> Вывод результата



In [ ]:
display(df_combined_high_low.head(200))
display(df_combined_high_low.tail(500))

,datetime,high_GAZP,low_GAZP,close_GAZP,high_TATN,low_TATN,close_TATN,high_CHMF,low_CHMF,close_CHMF,...,high_NLMK,low_NLMK,close_NLMK,high_ROSN,low_ROSN,close_ROSN,high_SNGS,low_SNGS,close_SNGS,GPRD
0,2008-05-23,2.175,2.113,2.140,188.0,184.1,187.5,605.0,584.8,594.6,...,418.0,365.0,367.0,270.97,261.71,269.82,29.450,28.590,29.115,88.375771
1,2008-05-26,2.169,2.129,2.143,189.5,181.1,184.0,594.6,587.6,593.6,...,414.0,365.0,398.0,276.30,268.20,275.15,29.285,28.465,28.720,88.375771
2,2008-05-27,2.151,2.095,2.120,186.5,181.2,182.1,597.0,583.8,589.2,...,400.0,400.0,400.0,281.65,270.50,272.75,29.515,27.545,27.675,83.478226
3,2008-05-30,2.149,2.100,2.115,189.9,184.8,189.4,632.4,610.4,617.6,...,399.0,300.0,370.0,287.03,278.63,286.78,29.268,27.811,29.063,79.820908
4,2008-06-03,2.185,2.113,2.177,188.5,185.0,186.9,612.6,590.8,596.0,...,411.0,378.0,378.0,287.65,282.60,283.75,28.952,28.420,28.767,110.962860
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,2010-11-17,1.568,1.548,1.552,149.5,146.1,148.9,439.6,428.6,436.0,...,950.0,900.0,950.0,213.50,210.60,212.05,29.794,29.454,29.679,77.664688
196,2010-11-22,1.585,1.572,1.584,151.5,150.0,150.9,462.2,450.6,451.8,...,900.0,900.0,900.0,216.90,212.25,213.45,30.105,29.870,29.900,113.150131
197,2010-11-23,1.614,1.571,1.613,150.9,148.8,150.5,450.6,441.2,443.6,...,900.0,900.0,900.0,212.17,209.42,210.57,29.985,29.620,29.685,77.379219
198,2010-12-06,1.679,1.632,1.674,154.0,150.0,154.0,498.7,485.1,490.7,...,908.0,908.0,908.0,219.56,215.81,219.16,30.961,30.336,30.921,110.682457


,datetime,high_GAZP,low_GAZP,close_GAZP,high_TATN,low_TATN,close_TATN,high_CHMF,low_CHMF,close_CHMF,...,high_NLMK,low_NLMK,close_NLMK,high_ROSN,low_ROSN,close_ROSN,high_SNGS,low_SNGS,close_SNGS,GPRD
1887,2022-01-17,0.7376,0.7207,0.7293,508.5,492.0,495.7,1568.8,1493.4,1514.8,...,2190.0,2105.0,2135.0,614.00,578.05,592.55,38.950,37.335,38.010,133.571274
1888,2022-01-18,0.7316,0.6850,0.7005,512.6,466.4,474.0,1521.8,1416.4,1460.4,...,2125.0,1930.0,1980.0,598.70,525.10,544.80,38.250,35.160,36.580,138.821487
1889,2022-01-19,0.7223,0.6788,0.7173,502.5,458.3,490.7,1496.8,1416.0,1480.6,...,2035.0,1940.0,1950.0,575.40,523.10,566.90,37.690,35.200,36.390,153.562744
1890,2022-01-20,0.7286,0.7068,0.7138,501.8,476.4,478.8,1524.8,1471.0,1482.8,...,2005.0,1970.0,1980.0,581.00,552.30,561.70,37.100,36.200,36.200,136.373978
1891,2022-01-24,0.7170,0.6658,0.7017,480.5,436.0,459.2,1492.6,1367.4,1416.0,...,1945.0,1740.0,1825.0,566.80,523.95,546.90,36.100,32.970,34.595,185.801071
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2382,2024-02-12,0.7692,0.7560,0.7667,715.2,705.1,711.6,1599.8,1567.2,1589.0,...,5600.0,5100.0,5500.0,592.35,585.00,588.95,30.360,29.995,30.280,156.723633
2383,2024-02-13,0.7730,0.7618,0.7703,718.0,711.3,712.1,1616.8,1589.0,1609.8,...,5800.0,5490.0,5790.0,593.45,589.55,592.75,30.735,30.000,30.500,187.858292
2384,2024-02-14,0.7706,0.7636,0.7654,713.9,707.1,707.7,1622.0,1605.4,1608.4,...,5800.0,5590.0,5600.0,594.65,589.65,590.05,30.930,30.350,30.400,135.809372
2385,2024-02-15,0.7670,0.7573,0.7601,711.5,706.4,709.3,1618.2,1591.6,1613.2,...,5640.0,5430.0,5590.0,591.60,587.55,589.15,30.870,30.265,30.780,214.808151




> Общая диагностика NaN в таблице



In [ ]:
print("Total rows:", len(df_combined_high_low))
print("Total NaN cells:", df_combined_high_low.isna().sum().sum())

Total rows: 2387
Total NaN cells: 0




> Подсчёт NaN по колонкам



In [ ]:
nan_by_column = df_combined_high_low.isna().sum()
nan_by_column = nan_by_column[nan_by_column > 0]

print("Columns with NaN:")
print(nan_by_column)

Columns with NaN:
Series([], dtype: int64)




>  Проверка NaN только в ценовых колонках



In [ ]:
price_cols = [c for c in df_combined_high_low.columns
              if c.startswith(("high_", "low_", "close_"))]

price_nan = df_combined_high_low[price_cols].isna().sum().sum()
print("Total NaN in price columns:", price_nan)

Total NaN in price columns: 0




> Проверка NaN конкретно в GPRD



In [ ]:
print("NaN in GPRD:", df_combined_high_low["GPRD"].isna().sum())

NaN in GPRD: 0




> Просмотр строк, где есть хотя бы один NaN


In [ ]:
rows_with_nan = df_combined_high_low[df_combined_high_low.isna().any(axis=1)]
display(rows_with_nan.head(100))

,datetime,high_GAZP,low_GAZP,close_GAZP,high_TATN,low_TATN,close_TATN,high_CHMF,low_CHMF,close_CHMF,...,high_NLMK,low_NLMK,close_NLMK,high_ROSN,low_ROSN,close_ROSN,high_SNGS,low_SNGS,close_SNGS,GPRD




> Заполнение пропусков GPRD значением предыдущего дня (forward fill) - убрать NaN в GPRD на выходных/праздниках, чтобы индикатор был определён на всех датах в таблице.



In [ ]:
df_combined_high_low = df_combined_high_low.sort_values("datetime").copy()
df_combined_high_low["GPRD"] = df_combined_high_low["GPRD"].ffill()



> NaN left in GPRD after ffill



In [ ]:
print("NaN in GPRD:", df_combined_high_low["GPRD"].isna().sum())

NaN in GPRD: 0




> Checking duplicates



In [ ]:
dupl_count = df_combined_high_low.duplicated(subset=["datetime"]).sum()
print("Duplicate datetime rows:", dupl_count)

Duplicate datetime rows: 0




> Финальный просмотр результата



In [ ]:
display(df_combined_high_low.head(20))
display(df_combined_high_low.tail(20))

,datetime,high_GAZP,low_GAZP,close_GAZP,high_TATN,low_TATN,close_TATN,high_CHMF,low_CHMF,close_CHMF,...,high_NLMK,low_NLMK,close_NLMK,high_ROSN,low_ROSN,close_ROSN,high_SNGS,low_SNGS,close_SNGS,GPRD
0,2008-05-23,2.175,2.113,2.140,188.0,184.1,187.5,605.0,584.8,594.6,...,418.0,365.0,367.0,270.97,261.71,269.82,29.450,28.590,29.115,88.375771
1,2008-05-26,2.169,2.129,2.143,189.5,181.1,184.0,594.6,587.6,593.6,...,414.0,365.0,398.0,276.30,268.20,275.15,29.285,28.465,28.720,88.375771
2,2008-05-27,2.151,2.095,2.120,186.5,181.2,182.1,597.0,583.8,589.2,...,400.0,400.0,400.0,281.65,270.50,272.75,29.515,27.545,27.675,83.478226
3,2008-05-30,2.149,2.100,2.115,189.9,184.8,189.4,632.4,610.4,617.6,...,399.0,300.0,370.0,287.03,278.63,286.78,29.268,27.811,29.063,79.820908
4,2008-06-03,2.185,2.113,2.177,188.5,185.0,186.9,612.6,590.8,596.0,...,411.0,378.0,378.0,287.65,282.60,283.75,28.952,28.420,28.767,110.962860
5,2008-06-05,2.166,2.113,2.113,183.4,176.0,182.8,598.8,584.8,587.0,...,400.0,321.0,340.0,277.70,268.00,273.75,28.160,27.232,27.785,69.199867
6,2008-06-09,2.079,1.950,2.061,186.0,180.0,184.4,582.8,568.1,578.4,...,332.0,332.0,332.0,281.91,272.85,278.50,28.090,27.070,27.720,69.983826
7,2008-06-18,2.070,2.015,2.028,186.3,182.2,183.0,613.0,578.2,602.6,...,308.0,308.0,308.0,286.39,281.39,282.64,28.335,27.805,27.805,82.584785
8,2008-06-20,2.027,1.975,1.994,184.5,180.5,181.8,623.8,613.0,618.4,...,345.0,271.0,300.0,280.74,275.24,275.64,28.017,27.302,27.527,130.369385
9,2008-06-24,2.040,1.973,1.990,183.9,176.7,176.8,621.8,593.6,595.8,...,300.0,300.0,300.0,271.68,262.98,264.18,27.176,26.251,26.326,164.259552


,datetime,high_GAZP,low_GAZP,close_GAZP,high_TATN,low_TATN,close_TATN,high_CHMF,low_CHMF,close_CHMF,...,high_NLMK,low_NLMK,close_NLMK,high_ROSN,low_ROSN,close_ROSN,high_SNGS,low_SNGS,close_SNGS,GPRD
2367,2024-01-22,0.7731,0.7561,0.7702,697.3,690.4,695.5,1619.6,1577.0,1619.4,...,5835.0,5700.0,5750.0,579.70,575.40,577.95,29.255,28.550,29.035,186.623535
2368,2024-01-23,0.7749,0.7647,0.7692,696.5,690.3,695.4,1624.8,1586.2,1604.4,...,5990.0,5725.0,5950.0,580.50,576.50,579.90,29.430,28.995,29.190,169.072449
2369,2024-01-24,0.7815,0.7641,0.7660,697.6,690.1,691.1,1619.6,1595.2,1614.0,...,6250.0,5950.0,6135.0,582.15,572.10,575.75,29.420,28.750,28.895,159.144714
2370,2024-01-25,0.7693,0.7531,0.7596,694.9,687.3,689.3,1623.8,1601.0,1609.0,...,6490.0,5780.0,6020.0,576.00,572.60,574.65,29.375,28.625,29.185,202.294205
2371,2024-01-26,0.7673,0.7500,0.7513,693.6,688.1,693.5,1615.6,1590.4,1602.0,...,6055.0,5900.0,5955.0,577.65,572.35,573.90,29.335,29.000,29.035,141.321152
2372,2024-01-29,0.7965,0.7517,0.7965,698.4,693.0,694.1,1630.6,1592.2,1629.8,...,6145.0,5780.0,5830.0,580.45,573.00,575.75,29.300,28.715,28.930,140.533173
2373,2024-01-30,0.7965,0.7781,0.7900,696.5,690.2,692.2,1685.0,1631.0,1683.8,...,5930.0,5835.0,5880.0,578.45,572.60,573.95,29.120,28.840,29.025,206.277115
2374,2024-01-31,0.7974,0.7802,0.7827,705.8,691.2,703.5,1688.8,1647.8,1664.0,...,5900.0,5780.0,5880.0,576.50,572.00,575.00,29.280,29.000,29.145,205.607651
2375,2024-02-01,0.7848,0.7688,0.7720,711.2,702.1,704.4,1679.8,1650.4,1672.0,...,5880.0,5720.0,5740.0,582.00,574.00,576.50,30.925,29.140,30.895,147.772537
2376,2024-02-02,0.7736,0.7651,0.7663,709.5,702.5,706.9,1710.0,1625.0,1627.8,...,5900.0,5350.0,5900.0,584.95,574.50,583.95,31.200,30.265,30.450,171.524551


In [ ]:
out_name = 'gpr_rus.csv'
df_combined_high_low.to_csv(out_name, index=False)
print('Saved:', out_name)


Saved: gpr_rus.csv


In [ ]:
files.download(out_name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>